In [2]:
#Importing Dependencies
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

In [3]:
#Loading Data
train_path = r"F:\3000 stuff\UNSW_NB15_training-set.csv"
test_path  = r"F:\3000 stuff\UNSW_NB15_testing-set.csv"

df_train = pd.read_csv(train_path).drop_duplicates().reset_index(drop=True)
df_test  = pd.read_csv(test_path).drop_duplicates().reset_index(drop=True)

df_train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [4]:
# Remove non-feature columns and prevent leaks
drop_cols = ["srcip", "dstip", "attack_cat", "id"]
df_train = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns], errors="ignore")
df_test  = df_test.drop(columns=[c for c in drop_cols if c in df_test.columns], errors="ignore")

y_train = df_train["label"].astype(int)
y_test  = df_test["label"].astype(int)
X_train = df_train.drop(columns=["label"])
X_test  = df_test.drop(columns=["label"])

In [5]:
#Data Preprocessing
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med).replace([np.inf, -np.inf], med)
    X_test[col]  = X_test[col].fillna(med).replace([np.inf, -np.inf], med)

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
X_test_enc  = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)

In [6]:
#Align train and test features
for col in set(X_train_enc.columns) - set(X_test_enc.columns):
    X_test_enc[col] = 0
X_test_enc = X_test_enc[X_train_enc.columns]

X_train_enc = X_train_enc.astype('float64')
X_test_enc  = X_test_enc.astype('float64')


In [7]:
#Class weights for XGBoost to ensure proper results
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos


In [8]:
#Model Training
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Logistic Regression": LogisticRegression(
        class_weight='balanced', max_iter=500, n_jobs=-1, random_state=42
    ),
    "Linear SVM": LinearSVC(class_weight='balanced', max_iter=5000, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist', random_state=42, n_jobs=-1
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train_enc, y_train)
    y_pred = model.predict(X_test_enc)
    
    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results[name] = {
        "Accuracy": acc,
        "Recall": recall,
        "Precision": precision,
        "F1": f1
    }


In [9]:
#Summary of results
results_sorted = dict(sorted(results.items(), key=lambda x: x[1]["Accuracy"], reverse=True))
print(f"\n{'Model':25s} | {'Accuracy':8s} | {'Recall':8s} | {'Precision':8s} | {'F1':8s}")
print("-"*70)
for name, metrics in results_sorted.items():
    print(f"{name:25s} | {metrics['Accuracy']:.4f}  | {metrics['Recall']:.4f}  | {metrics['Precision']:.4f}  | {metrics['F1']:.4f}")


Model                     | Accuracy | Recall   | Precision | F1      
----------------------------------------------------------------------
Random Forest             | 0.9099  | 0.9683  | 0.8801  | 0.9221
XGBoost                   | 0.9081  | 0.9646  | 0.8800  | 0.9203
KNN                       | 0.7855  | 0.9423  | 0.7395  | 0.8287
Logistic Regression       | 0.7277  | 0.7556  | 0.7513  | 0.7534
Linear SVM                | 0.7184  | 0.8712  | 0.6948  | 0.7731
